# Poker: reproducible baseline with fit diagnostics

**Kaggle instructions:** Add the competition data, select CPU, and Run All. No GPU or internet is required when the checked dependencies are present. Download `/kaggle/working/submission.csv` after all validation cells pass.

This is a **seat-based reference baseline**, not a high-score claim. It uses 5 table/player-disjoint folds, fixed LightGBM hyperparameters, official scoring, and an unlearned evidence heuristic. It never trains on evaluation labels or tunes on outer-fold results.

Overfitting/underfitting cannot be guaranteed absent. Train–validation gaps, frozen learning-curve diagnostics, a simpler model comparator, and a table bootstrap help identify concerns. If these results motivate tuning, use inner grouped CV and preserve a fresh outer assessment. The unlabeled-background score is contaminated, not a leaderboard forecast.


In [ ]:
from pathlib import Path
import os, sys, importlib.util, json, subprocess, shutil

required_modules = ['numpy', 'pandas', 'pyarrow', 'sklearn', 'lightgbm']
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(f"Missing dependencies: {missing}. Install these before Run All.")

START = Path.cwd()
KAGGLE = Path('/kaggle/input').exists()
required_files = ['hands.parquet', 'seats.parquet', 'development_labels.csv',
                  'development_evidence.csv', 'evaluation_pairs.csv', 'sample_submission.csv']
if KAGGLE:
    candidates = sorted({p.parent for p in Path('/kaggle/input').rglob('development_labels.csv')})
else:
    candidates = [START]
valid = [p for p in candidates if all((p / f).exists() for f in required_files)]
if len(valid) != 1:
    raise RuntimeError(f"Expected one competition folder, found {valid}. Add the competition data or set candidates explicitly.")
DATA_DIR = valid[0].resolve()
WORK = Path('/kaggle/working') if KAGGLE else START / 'artifacts/kaggle_notebook_validation'
WORK.mkdir(parents=True, exist_ok=True)
CODE_DIR = WORK / 'baseline_module'
CODE_DIR.mkdir(exist_ok=True)
OUT = WORK / 'artifacts/baseline'
os.chdir(WORK)
print('Data:', DATA_DIR, '\nWorking:', WORK)


## Included source

The next cells write the complete modules locally; no repository download is needed. The official metric comes from the supplied host notebook:
https://www.kaggle.com/code/florianderoofr/slash-poker-competition-metric

IDs are join/group keys only. Fixed features are symmetric seat outcomes/contributions in big blinds, fold/showdown rates, exposure, and flow imbalance. There are no learned evidence features, so this baseline has no evidence-stacking leakage path.


In [ ]:
%%writefile baseline_module/official_metric.py
# Copied unchanged from the supplied host metric notebook code cells.
import numpy as np
import pandas as pd


class ParticipantVisibleError(Exception):
    """An error message that Kaggle may safely show to participants."""


ALLOWED_BEHAVIORS = {
    "none",
    "directed_transfer",
    "soft_play",
    "coordinated_isolation",
    "other_coordination",
}
TARGET_BEHAVIORS = (
    "directed_transfer",
    "soft_play",
    "coordinated_isolation",
)
EVIDENCE_COLUMNS = tuple(f"evidence_hand_{rank}" for rank in range(1, 6))
NO_EVIDENCE = "NO_EVIDENCE"
REQUIRED_COLUMNS = {
    "pair_id",
    "risk_score",
    "predicted_behavior",
    *EVIDENCE_COLUMNS,
}


def _average_precision(y_true: np.ndarray, scores: np.ndarray) -> float:
    positives = int(y_true.sum())
    if positives == 0:
        return 0.0
    order = np.argsort(-scores, kind="mergesort")
    ranked = y_true[order]
    true_positives = np.cumsum(ranked)
    ranks = np.arange(1, len(ranked) + 1)
    return float(np.sum((true_positives / ranks) * ranked) / positives)


def _clean_evidence(values: list[object]) -> list[str]:
    cleaned = []
    for value in values:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if text and text != NO_EVIDENCE:
            cleaned.append(text)
    return cleaned


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
) -> float:
    """Score pair detection, behavior classification, and evidence retrieval.

    Final score:
      70% pair Average Precision
      20% evidence MAP@5
      10% behavior macro Average Precision, always averaged over the three
      disclosed families; an absent family contributes zero

    Higher is better. The maximum score is 1.
    """

    if row_id_column_name != "pair_id":
        raise ParticipantVisibleError("The row ID column must be pair_id.")
    if not REQUIRED_COLUMNS.issubset(solution.columns):
        raise ValueError("The private solution has an invalid schema.")
    if not REQUIRED_COLUMNS.issubset(submission.columns):
        missing = sorted(REQUIRED_COLUMNS - set(submission.columns))
        raise ParticipantVisibleError(
            f"submission.csv is missing columns: {missing}"
        )
    if solution["pair_id"].duplicated().any():
        raise ValueError("The private solution contains duplicate pair IDs.")
    if submission["pair_id"].duplicated().any():
        raise ParticipantVisibleError("pair_id values must be unique.")

    solution_ids = set(solution["pair_id"].astype(str))
    submission_ids = set(submission["pair_id"].astype(str))
    if solution_ids != submission_ids:
        missing = len(solution_ids - submission_ids)
        extra = len(submission_ids - solution_ids)
        raise ParticipantVisibleError(
            f"pair_id coverage mismatch: {missing} missing and {extra} extra."
        )

    truth = solution.set_index("pair_id").sort_index()
    predictions = submission.set_index("pair_id").loc[truth.index]

    risk = pd.to_numeric(predictions["risk_score"], errors="coerce")
    if risk.isna().any() or not risk.between(0, 1).all():
        raise ParticipantVisibleError(
            "risk_score must be numeric and between 0 and 1."
        )

    predicted_behavior = predictions["predicted_behavior"].astype(str)
    invalid = set(predicted_behavior) - ALLOWED_BEHAVIORS
    if invalid:
        raise ParticipantVisibleError(
            f"Invalid predicted_behavior values: {sorted(invalid)}"
        )

    for row in predictions.loc[:, EVIDENCE_COLUMNS].itertuples(
        index=False,
        name=None,
    ):
        evidence = _clean_evidence(list(row))
        if len(evidence) != len(set(evidence)):
            raise ParticipantVisibleError(
                "Evidence hand IDs must not repeat within a pair."
            )

    y_true = pd.to_numeric(truth["risk_score"], errors="raise").to_numpy(
        dtype=int
    )
    if not set(np.unique(y_true)).issubset({0, 1}):
        raise ValueError("Private risk_score values must be binary labels.")
    risk_values = risk.to_numpy(dtype=float)
    pair_ap = _average_precision(y_true, risk_values)

    true_behavior = truth["predicted_behavior"].astype(str).to_numpy()
    predicted_behavior_values = predicted_behavior.to_numpy()
    behavior_scores = []
    for behavior in TARGET_BEHAVIORS:
        behavior_truth = (true_behavior == behavior).astype(int)
        if behavior_truth.sum() == 0:
            behavior_scores.append(0.0)
            continue
        behavior_risk = np.where(
            predicted_behavior_values == behavior,
            risk_values,
            0.0,
        )
        behavior_scores.append(
            _average_precision(behavior_truth, behavior_risk)
        )
    behavior_map = float(np.mean(behavior_scores))

    evidence_scores = []
    for position in np.flatnonzero(y_true == 1):
        relevant = set(
            _clean_evidence(
                truth.iloc[position].loc[list(EVIDENCE_COLUMNS)].tolist()
            )
        )
        submitted = _clean_evidence(
            predictions.iloc[position]
            .loc[list(EVIDENCE_COLUMNS)]
            .tolist()
        )
        if not relevant:
            evidence_scores.append(0.0)
            continue
        hits = 0
        precision_sum = 0.0
        for rank, hand_id in enumerate(submitted[:5], start=1):
            if hand_id in relevant:
                hits += 1
                precision_sum += hits / rank
        evidence_scores.append(precision_sum / min(len(relevant), 5))
    evidence_map = (
        float(np.mean(evidence_scores)) if evidence_scores else 0.0
    )

    final_score = (
        0.70 * pair_ap
        + 0.20 * evidence_map
        + 0.10 * behavior_map
    )
    if not np.isfinite(final_score):
        raise ParticipantVisibleError("The metric produced a non-finite score.")
    return float(final_score)


In [ ]:
%%writefile baseline_module/run.py
"""Fixed, table-held-out P/N baseline; deterministic, unlearned evidence retrieval."""
from pathlib import Path
import argparse
import hashlib
import itertools
import json
import inspect
import platform
import time

import numpy as np
import pandas as pd
import pyarrow
import sklearn
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score
from official_metric import score, _average_precision, EVIDENCE_COLUMNS

FAMILIES = ['directed_transfer', 'soft_play', 'coordinated_isolation']
SEED = 2026


def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def evidence_predictions(candidates):
    """Return top-five without labels, model predictions, or risk gating."""
    ranked = candidates.sort_values(['key', 'heuristic', 'hand_id'], ascending=[True, False, True])
    ranked = ranked.drop_duplicates(['key', 'hand_id']).groupby('key', sort=False).head(5).copy()
    ranked['slot'] = ranked.groupby('key').cumcount() + 1
    wide = ranked.pivot(index='key', columns='slot', values='hand_id').reindex(columns=range(1, 6))
    wide.columns = list(EVIDENCE_COLUMNS)
    return wide.fillna('NO_EVIDENCE').reset_index()


def build_features(root, out):
    print('Loading selected hand/seat columns', flush=True)
    hands = pd.read_parquet(root/'hands.parquet')
    seats = pd.read_parquet(root/'seats.parquet', columns=[
        'hand_id', 'player_id', 'net_chips', 'total_contribution', 'folded', 'went_to_showdown'])
    players = pd.Index(sorted(seats.player_id.unique()))
    nplayers = len(players)
    hid = pd.Index(hands.hand_id)
    assert hid.is_unique
    hi = hid.get_indexer(seats.hand_id)
    pi = players.get_indexer(seats.player_id)
    assert (hi >= 0).all() and (pi >= 0).all()
    order = np.lexsort((pi, hi))
    assert len(seats) == 6 * len(hands)
    assert np.array_equal(np.bincount(hi), np.full(len(hands), 6))
    p = pi[order].reshape(-1, 6)
    assert (np.diff(p, axis=1) > 0).all()
    arrays = {c: seats[c].to_numpy()[order].reshape(-1, 6) for c in
              ['net_chips', 'total_contribution', 'folded', 'went_to_showdown']}
    assert (arrays['net_chips'].sum(axis=1) == 0).all()
    membership = pd.DataFrame({'player': p.ravel(), 'table': np.repeat(hands.table_id.to_numpy(), 6)})
    assert membership.groupby('player').table.nunique().eq(1).all()
    del seats, hi, pi, order, membership
    labels = pd.read_csv(root/'development_labels.csv')
    evaluation = pd.read_csv(root/'evaluation_pairs.csv')
    for frame in [labels, evaluation]:
        a, b = players.get_indexer(frame.player_1), players.get_indexer(frame.player_2)
        assert (a >= 0).all() and (b >= 0).all()
        frame['key'] = np.minimum(a, b).astype(np.int64)*nplayers + np.maximum(a, b)
        assert frame.key.is_unique
    labels.to_parquet(out/'labels.parquet', index=False)
    evaluation.to_parquet(out/'evaluation.parquet', index=False)
    eval_keys = set(evaluation.key)
    dev_keys = set(labels.key)
    i, j = np.array(list(itertools.combinations(range(6), 2))).T
    feats, evidence = [], []
    for ti, (table, indices) in enumerate(hands.groupby('table_id', sort=True).indices.items()):
        for phase in ['development', 'evaluation']:
            ix = indices[hands.phase.to_numpy()[indices] == phase]
            bb = hands.big_blind.to_numpy()[ix, None]
            net = arrays['net_chips'][ix]/bb
            contrib = arrays['total_contribution'][ix]/bb
            pl = p[ix]
            a, b = net[:, i], net[:, j]
            ca, cb = contrib[:, i], contrib[:, j]
            f = arrays['folded'][ix]
            sd = arrays['went_to_showdown'][ix]
            t12 = np.minimum(np.maximum(-a, 0), np.maximum(b, 0))
            t21 = np.minimum(np.maximum(-b, 0), np.maximum(a, 0))
            values = {
                'flow12': t12, 'flow21': t21,
                'transfer': t12+t21, 'net_gap': np.abs(a-b),
                'joint_net': a+b, 'contrib_gap': np.abs(ca-cb),
                'min_contrib': np.minimum(ca, cb), 'pair_contrib': ca+cb,
                'opposite_net': a*b < 0, 'both_showdown': sd[:, i] & sd[:, j],
                'both_fold': f[:, i] & f[:, j], 'one_fold': f[:, i] ^ f[:, j],
                'both_invested': (ca > 1) & (cb > 1),
            }
            frame = pd.DataFrame({k: v.ravel().astype(np.float32) for k,v in values.items()})
            frame['key'] = (pl[:, i].astype(np.int64)*nplayers + pl[:, j]).ravel()
            group = frame.groupby('key', sort=True)
            agg = group[list(values)].agg(['mean','std','max'])
            agg.columns = [f'{a}_{b}' for a,b in agg.columns]
            agg['shared_hands'] = group.size()
            agg['direction_imbalance'] = (agg.flow12_mean-agg.flow21_mean).abs()/(agg.transfer_mean+1)
            # Raw direction slots depend on arbitrary ID ordering; keep only symmetric summaries.
            agg = agg.drop(columns=[c for c in agg if c.startswith(('flow12_', 'flow21_'))])
            agg['table_id'] = table
            agg['phase'] = phase
            agg = agg.fillna(0).reset_index()
            if phase == 'development':
                agg = agg[(agg.shared_hands >= 57) | agg.key.isin(dev_keys)]
            else:
                agg = agg[agg.key.isin(eval_keys)]
            feats.append(agg)
            frame['hand_id'] = np.repeat(hands.hand_id.to_numpy()[ix], 15)
            # Fixed initial heuristic, deliberately not tuned on evidence labels.
            frame['heuristic'] = (np.log1p(frame.transfer) + .25*np.log1p(frame.pair_contrib)
                                  + .5*frame.both_showdown + .25*frame.both_invested)
            ev = evidence_predictions(frame[frame.key.isin(agg.key)][['key','hand_id','heuristic']])
            ev['phase'] = phase
            evidence.append(ev)
        if (ti+1) % 40 == 0:
            print(f'Features/evidence: {ti+1}/400 tables', flush=True)
    pd.concat(feats, ignore_index=True).to_parquet(out/'features.parquet', index=False)
    pd.concat(evidence, ignore_index=True).to_parquet(out/'evidence.parquet', index=False)


def predictions(rows, risk, behavior, ev):
    result = rows[['pair_id','key']].copy()
    result['risk_score'] = risk
    result['predicted_behavior'] = behavior
    return result.merge(ev.drop(columns='phase'), on='key', validate='one_to_one').drop(columns='key')


def truth_for(rows, evidence):
    truth = rows[['pair_id','label','behavior_family']].rename(
        columns={'label':'risk_score','behavior_family':'predicted_behavior'}).copy()
    wide = evidence.pivot(index='pair_id',columns='evidence_rank',values='hand_id').reindex(columns=range(1,6))
    wide.columns = list(EVIDENCE_COLUMNS)
    return truth.merge(wide, on='pair_id', how='left').fillna('NO_EVIDENCE')


def components(truth, pred):
    total = score(truth, pred, 'pair_id')
    t = truth.set_index('pair_id').sort_index()
    s = pred.set_index('pair_id').loc[t.index]
    pair = _average_precision(t.risk_score.to_numpy(), s.risk_score.to_numpy())
    behavior = np.mean([_average_precision((t.predicted_behavior == f).to_numpy(),
                        np.where(s.predicted_behavior == f, s.risk_score, 0)) for f in FAMILIES])
    return {'pair_ap':float(pair),'evidence_map5':float((total-.7*pair-.1*behavior)/.2),
            'behavior_map':float(behavior),'composite':float(total)}


def model(multiclass=False):
    return lgb.LGBMClassifier(objective='multiclass' if multiclass else 'binary',
        n_estimators=250, learning_rate=.03, num_leaves=15, max_depth=5,
        min_child_samples=25, reg_lambda=5., colsample_bytree=.8,
        random_state=SEED, n_jobs=4, verbosity=-1, deterministic=True, force_col_wise=True)


def train(root, out):
    all_f = pd.read_parquet(out/'features.parquet')
    labels = pd.read_parquet(out/'labels.parquet')
    evaluation = pd.read_parquet(out/'evaluation.parquet')
    ev = pd.read_parquet(out/'evidence.parquet')
    dev = all_f[all_f.phase.eq('development')].merge(labels[['key','pair_id','label','behavior_family']],on='key',how='left')
    known = dev.label.notna()
    # Synthetic diagnostic IDs are keys only, never model inputs.
    dev['pair_id'] = dev.pair_id.fillna('UNLABELED_'+dev.key.astype(str))
    dev['label'] = dev.label.fillna(0).astype(int)
    dev['behavior_family'] = dev.behavior_family.fillna('none')
    features = [c for c in all_f if c not in ['key','table_id','phase']]
    # Define outer fold assignment from trusted labels, then map entire tables.
    trusted = dev[known]
    splitter = StratifiedGroupKFold(5,shuffle=True,random_state=SEED)
    mapping = {}
    for fold,(_, va) in enumerate(splitter.split(trusted, trusted.behavior_family, trusted.table_id)):
        mapping.update({table:fold for table in trusted.iloc[va].table_id.unique()})
    for table in sorted(set(dev.table_id)-set(mapping)):
        mapping[table] = len(mapping) % 5
    dev['fold'] = dev.table_id.map(mapping)
    risk = np.zeros(len(dev))
    behavior = np.full(len(dev),'none',dtype=object)
    folds=[]
    for fold in range(5):
        tr = known & dev.fold.ne(fold)
        va = dev.fold.eq(fold)
        assert set(dev.loc[tr,'table_id']).isdisjoint(dev.loc[va,'table_id'])
        assert set(dev.loc[tr & dev.label.eq(1),'behavior_family']) == set(FAMILIES)
        m=model();m.fit(dev.loc[tr,features],dev.loc[tr,'label'])
        risk[va]=m.predict_proba(dev.loc[va,features])[:,1]
        pos=tr & dev.label.eq(1)
        b=model(True);b.fit(dev.loc[pos,features],dev.loc[pos,'behavior_family'])
        behavior[va]=b.predict(dev.loc[va,features])
        subset=dev[va & known]
        pr=predictions(subset,risk[va & known],behavior[va & known],ev[ev.phase.eq('development')])
        met=components(truth_for(subset,pd.read_csv(root/'development_evidence.csv')),pr)
        folds.append({'fold':fold,'train_labeled':int(tr.sum()),'valid_labeled':len(subset),**met})
        print('Fold',fold,met,flush=True)
    pr=predictions(dev,risk,behavior,ev[ev.phase.eq('development')])
    pr.to_csv(out/'oof_predictions.csv',index=False)
    dev[['pair_id','table_id','fold']].assign(is_labeled=known).to_csv(out/'folds.csv',index=False)
    t=truth_for(dev,pd.read_csv(root/'development_evidence.csv'))
    metrics={'confirmed_labels':components(t[known.to_numpy()],pr[pr.pair_id.isin(dev.loc[known,'pair_id'])]),
             'unlabeled_as_negative_stress':{
                 'pair_ap':float(average_precision_score(dev.label,risk)),
                 'ap_definition':'sklearn threshold-grouped AP; unknown development pair IDs unavailable',
                 'behavior_and_composite':'Not reported: private labels and original unlabeled IDs unavailable'},'folds':folds,
             'limitations':['Stress labels are contaminated, not evaluation ground truth.',
               'Fixed seat-only heuristic evidence; no learned evidence model or stacking.',
               'No hidden-family detector; behavior always routes to a known family.',
               'Fixed hyperparameters, no outer-fold early stopping or threshold selection.'],
             'feature_columns':features}
    m=model();m.fit(dev.loc[known,features],dev.loc[known,'label'])
    b=model(True);pos=known & dev.label.eq(1);b.fit(dev.loc[pos,features],dev.loc[pos,'behavior_family'])
    m.booster_.save_model(str(out/'risk_model.txt'));b.booster_.save_model(str(out/'behavior_model.txt'))
    exposure=evaluation[['key','shared_hands']].merge(
        all_f[all_f.phase.eq('evaluation')][['key','shared_hands']],on='key',suffixes=('_given','_computed'),validate='one_to_one')
    assert len(exposure)==len(evaluation) and exposure.shared_hands_given.eq(exposure.shared_hands_computed).all()
    test=evaluation.merge(all_f[all_f.phase.eq('evaluation')].drop(columns='shared_hands'),on='key',validate='one_to_one')
    sub=predictions(test,m.predict_proba(test[features])[:,1],b.predict(test[features]),ev[ev.phase.eq('evaluation')])
    sample=pd.read_csv(root/'sample_submission.csv')
    sub=sample[['pair_id']].merge(sub,on='pair_id',validate='one_to_one')[sample.columns]
    sub.to_csv(out/'submission.csv',index=False,float_format='%.17g')
    validate(root,out)
    metrics['rows']={'labeled':int(known.sum()),'unlabeled_stress':int((~known).sum()),'evaluation':len(sub)}
    (out/'metrics.json').write_text(json.dumps(metrics,indent=2)+'\n')
    c=metrics['confirmed_labels'];stress=metrics['unlabeled_as_negative_stress']['pair_ap']
    report=['# Baseline validation report','',
        'Measured locally. Five table-disjoint outer folds; fixed hyperparameters; trusted P/N training only.',
        'Evidence is a fixed seat-based heuristic. There is no learned evidence stacking, threshold tuning, or early stopping.',
        '', '| Population / metric | Value |','|---|---:|',
        *[f'| Confirmed-label {k} | {v:.6f} |' for k,v in c.items()],
        f'| Unlabeled-as-negative stress pair AP (sklearn) | {stress:.6f} |','',
        '## Outer folds','', '| Fold | Training labels | Validation labels | Pair AP | Evidence MAP@5 | Behavior MAP | Composite |',
        '|---:|---:|---:|---:|---:|---:|---:|',
        *[f"| {f['fold']} | {f['train_labeled']} | {f['valid_labeled']} | {f['pair_ap']:.6f} | {f['evidence_map5']:.6f} | {f['behavior_map']:.6f} | {f['composite']:.6f} |" for f in folds],
        '', '## Interpretation','',
        'The labeled composite is a development benchmark, not a leaderboard forecast. The stress set includes all eligible development background pairs, including pairs involving positive players; it does not exactly reproduce evaluation selection. Unknown pairs are provisionally negative, so stress AP is contaminated. No stress behavior/composite is reported.',
        'All behavior assignments use held-out predictions. All three known families are eligible at any risk; no hidden-family detector is implemented.',
        'The evidence heuristic emphasizes chip-flow proxies and may miss soft play and isolation. Net-flow proxies do not identify exact bilateral transfers in multiway pots.',
        '', '## Submission checks','',
        f'Passed: {len(sub):,} unique required pairs; finite risks in [0,1]; valid behaviors; five distinct shared evaluation hands per pair; recomputed exposure equals the supplied shared_hands.',
        '', 'No Kaggle upload was performed. See manifest.json for source/data hashes and versions.']
    (out/'REPORT.md').write_text('\n'.join(report)+'\n')
    return metrics


def validate(root,out):
    sub=pd.read_csv(out/'submission.csv',keep_default_na=False)
    pairs=pd.read_csv(root/'evaluation_pairs.csv')
    assert len(sub)==len(pairs) and sub.pair_id.is_unique and set(sub.pair_id)==set(pairs.pair_id)
    assert sub[list(EVIDENCE_COLUMNS)].ne('NO_EVIDENCE').all().all()
    fake=sub.copy();fake.risk_score=0;fake.predicted_behavior='none'
    score(fake,sub,'pair_id')
    long=sub.melt(id_vars='pair_id',value_vars=list(EVIDENCE_COLUMNS),value_name='hand_id')
    hands=pd.read_parquet(root/'hands.parquet',columns=['hand_id','phase'])
    assert long.merge(hands,on='hand_id',validate='many_to_one').phase.eq('evaluation').all()
    assert long.hand_id.isin(hands.hand_id).all()
    seats=pd.read_parquet(root/'seats.parquet',columns=['hand_id','player_id'])
    lookup=pd.MultiIndex.from_frame(seats)
    target=long.merge(pairs,on='pair_id',validate='many_to_one')
    for side in ['player_1','player_2']:
        query=pd.MultiIndex.from_arrays([target.hand_id,target[side]])
        assert (lookup.get_indexer(query)>=0).all()
    print(f'Validated {len(sub):,} pairs and {len(long):,} shared evaluation evidence entries',flush=True)


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--rebuild',action='store_true')
    parser.add_argument('--data-dir',type=Path,default=Path(__file__).resolve().parents[1])
    parser.add_argument('--output-dir',type=Path)
    args=parser.parse_args()
    root=args.data_dir.resolve();out=(args.output_dir or root/'artifacts/baseline').resolve()
    out.mkdir(parents=True,exist_ok=True)
    start=time.time()
    inputs=['hands.parquet','seats.parquet','development_labels.csv','development_evidence.csv','evaluation_pairs.csv','sample_submission.csv']
    fingerprint={f:digest(root/f) for f in inputs}
    feature_source=inspect.getsource(build_features)+inspect.getsource(evidence_predictions)
    cache_signature={'inputs':fingerprint,'feature_code_sha256':hashlib.sha256(feature_source.encode()).hexdigest()}
    cache=out/'cache_signature.json'
    if args.rebuild or not cache.exists() or json.loads(cache.read_text())!=cache_signature:
        build_features(root,out);cache.write_text(json.dumps(cache_signature,indent=2)+'\n')
    result=train(root,out)
    manifest={**cache_signature,'pipeline_sha256':digest(Path(__file__)), 'official_metric_sha256':digest(Path(__file__).with_name('official_metric.py')),
              'seed':SEED,'elapsed_seconds':time.time()-start,'python':platform.python_version(),
              'versions':{x.__name__:x.__version__ for x in [np,pd,pyarrow,sklearn,lgb]},
              'submission_sha256':digest(out/'submission.csv')}
    (out/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
    print(json.dumps(result,indent=2),flush=True)


if __name__=='__main__':
    main()


In [ ]:
%%writefile baseline_module/test_baseline.py
import unittest
import numpy as np
import pandas as pd
from official_metric import score, _average_precision, EVIDENCE_COLUMNS, ParticipantVisibleError
from run import evidence_predictions, components


class BaselineChecks(unittest.TestCase):
    def frames(self):
        truth=pd.DataFrame({'pair_id':['P1','P2'],'risk_score':[1,0],
                            'predicted_behavior':['soft_play','none']})
        for i,c in enumerate(EVIDENCE_COLUMNS):
            truth[c]=['H1' if i==0 else 'NO_EVIDENCE','NO_EVIDENCE']
        pred=truth.copy();pred.risk_score=[.8,.2]
        return truth,pred

    def test_exact_ties_and_components(self):
        self.assertEqual(_average_precision(np.array([1,0]),np.zeros(2)),1.)
        t,p=self.frames();m=components(t,p)
        self.assertAlmostEqual(m['evidence_map5'],1.)
        self.assertAlmostEqual(m['composite'],score(t,p,'pair_id'))

    def test_duplicate_evidence_rejected(self):
        t,p=self.frames();p.loc[0,'evidence_hand_2']='H1'
        with self.assertRaises(ParticipantVisibleError):score(t,p,'pair_id')

    def test_evidence_ranking_and_padding(self):
        c=pd.DataFrame({'key':[1,1,1,2],'hand_id':['H1','H2','H2','H3'],
                        'heuristic':[1.,3.,2.,0.]})
        r=evidence_predictions(c).set_index('key')
        self.assertEqual(r.loc[1,'evidence_hand_1'],'H2')
        self.assertEqual(r.loc[1,'evidence_hand_2'],'H1')
        self.assertEqual(r.loc[1,'evidence_hand_3'],'NO_EVIDENCE')


if __name__=='__main__':unittest.main()


In [ ]:
%%writefile baseline_module/diagnostics.py
"""Frozen diagnostic comparisons. Never select a model using these outer-fold results."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from run import model, SEED
from official_metric import _average_precision


def diagnose(out):
    out=Path(out)
    features=pd.read_parquet(out/'features.parquet')
    labels=pd.read_parquet(out/'labels.parquet')
    folds=pd.read_csv(out/'folds.csv')
    dev=labels.merge(features[features.phase.eq('development')],on='key',validate='one_to_one')
    dev=dev.merge(folds[['pair_id','fold']],on='pair_id',validate='one_to_one').sort_values('pair_id').reset_index(drop=True)
    columns=json.loads((out/'metrics.json').read_text())['feature_columns']
    oof=pd.read_csv(out/'oof_predictions.csv').set_index('pair_id').loc[dev.pair_id]
    reports=[]
    for fold in range(5):
        tr=dev.fold.ne(fold);va=~tr
        assert set(dev.loc[tr,'table_id']).isdisjoint(dev.loc[va,'table_id'])
        training_players=set(dev.loc[tr,'player_1'])|set(dev.loc[tr,'player_2'])
        validation_players=set(dev.loc[va,'player_1'])|set(dev.loc[va,'player_2'])
        assert training_players.isdisjoint(validation_players)
        # Preserve original training row order to reproduce the frozen baseline exactly.
        train_order=dev.loc[tr].sort_values(['table_id','key'])
        X=train_order[columns];y=train_order.label
        m=model();m.fit(X,y)
        for rounds in [50,125,250]:
            ptr=m.predict_proba(X,num_iteration=rounds)[:,1]
            pva=m.predict_proba(dev.loc[va,columns],num_iteration=rounds)[:,1]
            if rounds==250:
                assert np.allclose(pva,oof.risk_score.to_numpy()[va],atol=1e-12,rtol=1e-10), 'Diagnostic refit differs from frozen OOF predictions'
            # Exact host tie order for training rows too.
            perm=np.argsort(train_order.pair_id.to_numpy(),kind='stable')
            train_ap=_average_precision(y.to_numpy()[perm],ptr[perm])
            val_ap=_average_precision(dev.loc[va,'label'].to_numpy(),pva)
            reports.append({'fold':fold,'model':'LightGBM','trees':rounds,
                'train_ap':train_ap,'validation_ap':val_ap,'ap_gap':train_ap-val_ap,
                'train_logloss':log_loss(y,ptr,labels=[0,1]),
                'validation_logloss':log_loss(dev.loc[va,'label'],pva,labels=[0,1]),
                'validation_prevalence':float(dev.loc[va,'label'].mean())})
        simple=make_pipeline(StandardScaler(),LogisticRegression(C=.1,max_iter=3000,random_state=SEED))
        simple.fit(X,y)
        ptr=simple.predict_proba(X)[:,1];pva=simple.predict_proba(dev.loc[va,columns])[:,1]
        train_ap=_average_precision(y.to_numpy()[perm],ptr[perm])
        val_ap=_average_precision(dev.loc[va,'label'].to_numpy(),pva)
        reports.append({'fold':fold,'model':'LogisticRegression','trees':0,
            'train_ap':train_ap,'validation_ap':val_ap,'ap_gap':train_ap-val_ap,
            'train_logloss':log_loss(y,ptr,labels=[0,1]),
            'validation_logloss':log_loss(dev.loc[va,'label'],pva,labels=[0,1]),
            'validation_prevalence':float(dev.loc[va,'label'].mean())})
    frame=pd.DataFrame(reports);frame.to_csv(out/'fit_diagnostics.csv',index=False)
    summary=frame.groupby(['model','trees'])[['train_ap','validation_ap','ap_gap','train_logloss','validation_logloss']].mean().reset_index()
    summary.to_csv(out/'fit_summary.csv',index=False)
    # Descriptive table bootstrap of fixed OOF scores; no model refitting.
    table_rows=[g.index.to_numpy() for _,g in dev.groupby('table_id',sort=True)]
    rng=np.random.default_rng(SEED);boot=[]
    for _ in range(300):
        ix=np.sort(np.concatenate([table_rows[i] for i in rng.integers(len(table_rows),size=len(table_rows))]))
        boot.append(_average_precision(dev.label.to_numpy()[ix],oof.risk_score.to_numpy()[ix]))
    final=summary[(summary.model=='LightGBM') & summary.trees.eq(250)].iloc[0]
    earlier=summary[(summary.model=='LightGBM') & summary.trees.eq(125)].iloc[0]
    alerts=[]
    if final.ap_gap>.10:alerts.append('Train/validation AP gap exceeds 0.10: investigate overfitting or distribution differences.')
    if final.validation_ap < earlier.validation_ap and final.train_ap > earlier.train_ap:
        alerts.append('More trees improve training AP but reduce held-out AP: a possible overfitting signal.')
    if final.validation_ap <= summary[summary.model=='LogisticRegression'].validation_ap.iloc[0]:
        alerts.append('LightGBM does not beat the simpler comparator on mean fold AP; investigate features/capacity.')
    alerts.append('These checks cannot certify absence of overfitting or underfitting; outer results are diagnostics, not tuning targets.')
    report={'table_and_player_overlap':0,'diagnostics_only_no_model_selection':True,
        'fixed_oof_pair_ap_table_bootstrap_95_percentile':np.quantile(boot,[.025,.975]).tolist(),
        'bootstrap_note':'Descriptive 300-table-bootstrap replicates of fixed OOF predictions, not private-LB uncertainty or full training uncertainty.',
        'alerts':alerts,'summary':summary.to_dict('records')}
    (out/'validation_diagnostics.json').write_text(json.dumps(report,indent=2)+'\n')
    print(summary.to_string(index=False));print('\n'.join(alerts))
    return frame,report


## Run metric/retrieval checks, then full pipeline

Fixed model: 250 trees, learning rate 0.03, depth 5, 15 leaves, L2 regularization 5. No early stopping against the outer fold. The behavior classifier uses only training-fold positive labels; the reported score never uses oracle behavior assignments.


In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(CODE_DIR), '-p', 'test_*.py'], check=True)
subprocess.run([sys.executable, str(CODE_DIR / 'run.py'),
                '--data-dir', str(DATA_DIR), '--output-dir', str(OUT)], check=True)


## Inspect actual out-of-fold results

The confirmed-label composite uses the exact host scorer. Broader background AP treats unknown relationships as negatives and uses threshold-grouped AP because original IDs for unknown development pairs are unavailable. Do not compare its value directly with the official leaderboard.


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, FileLink
metrics = json.loads((OUT / 'metrics.json').read_text())
display(pd.DataFrame([metrics['confirmed_labels']]))
display(pd.DataFrame(metrics['folds']))
print('Unlabeled-background diagnostic:', metrics['unlabeled_as_negative_stress'])
display(Markdown((OUT / 'REPORT.md').read_text()))


## Overfitting and underfitting diagnostics

Refit the **same frozen** outer models and inspect their predictions after 50, 125, and 250 trees. These results do not select a replacement model. Training and validation AP/log-loss are reported separately. A regularized logistic regression serves as a simpler comparator.

Large training/validation gaps suggest overfitting or population differences. Poor training and validation performance can indicate weak features or insufficient capacity, but AP depends on prevalence and neither pattern proves a unique cause. Log-loss is descriptive here because risk is trained on curated labels rather than evaluation prevalence.

The confidence interval resamples tables from fixed OOF predictions; it is not a private-leaderboard uncertainty estimate. Repeated tuning against these outer results would overfit the validation set.


In [ ]:
sys.path.insert(0, str(CODE_DIR))
from diagnostics import diagnose
fit_rows, diagnostics = diagnose(OUT)
display(pd.DataFrame(diagnostics['summary']))
print('Fixed-OOF table-bootstrap 95% interval:', diagnostics['fixed_oof_pair_ap_table_bootstrap_95_percentile'])
print('Player/table overlap:', diagnostics['table_and_player_overlap'])
for alert in diagnostics['alerts']:
    print('Diagnostic:', alert)


## Export only after all checks pass

The pipeline has checked every pair ID, finite score, allowed behavior, duplicate evidence, both-player membership, evaluation phase, and recomputed exposure. The file contains five shared evaluation hands for every pair.

Fit diagnostics may flag genuine limitations; they do not silently tune the submission or certify that it will generalize. This initial baseline lacks action-context, learned evidence retrieval, and a fourth-family detector.


In [ ]:
submission_path = WORK / 'submission.csv'
shutil.copyfile(OUT / 'submission.csv', submission_path)
from run import digest
manifest = json.loads((OUT / 'manifest.json').read_text())
assert digest(submission_path) == manifest['submission_sha256']
print('Ready:', submission_path)
print('Rows:', len(pd.read_csv(submission_path)))
print('SHA256:', manifest['submission_sha256'])
display(FileLink('submission.csv'))
display(FileLink('artifacts/baseline/metrics.json'))
display(FileLink('artifacts/baseline/validation_diagnostics.json'))
display(FileLink('artifacts/baseline/fit_diagnostics.csv'))
